# Wandaa Brain production notebook (Colab/Kaggle)

Requirements:
- GPU runtime
- Internet enabled
- Secret vault entries: `HF_TOKEN` (required), `MISTRAL_API_KEY` (required only when running `synth`)
- Optional private clone URL with GitHub token

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = os.environ.get("KINYA_REPO_URL", "https://github.com/map-boy/kinya.git")
ROOT = Path("/content/kinya" if Path("/content").exists() else "/kaggle/working/kinya")

if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "requests", "datasets", "huggingface_hub", "unsloth", "trl"], check=True)
print(f"repo: {ROOT}")

In [ ]:
from kinya.secrets import get_secret

hf = get_secret("HF_TOKEN") or get_secret("HUGGINGFACE_HUB_TOKEN")
if not hf:
    raise RuntimeError("HF token missing: add HF_TOKEN in Colab/Kaggle secrets vault")

mistral = get_secret("MISTRAL_API_KEY")
print("HF token loaded:", bool(hf), "length:", len(hf))
print("Mistral key loaded:", bool(mistral), "length:", len(mistral or ""))

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. Use a GPU runtime for train/eval.")
print("CUDA device:", torch.cuda.get_device_name(0))

In [ ]:
import json, subprocess, sys
from pathlib import Path

def run_stage(*args):
    cmd = [sys.executable, "-m", "kinya", "run", *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

subprocess.run([sys.executable, "-m", "kinya", "status"], check=True)
run_stage("pull")
run_stage("train")
run_stage("eval")
run_stage("push")

latest = Path("reports/latest.json")
if latest.exists():
    rep = json.loads(latest.read_text(encoding="utf-8"))
    print(json.dumps(rep, ensure_ascii=False, indent=2)[:4000])
else:
    print("No report produced")